In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Any

BASE_PATH = "shared/UCI-Benchmark/ds/"
UNI_DIR = os.path.join(BASE_PATH, "univariate")
MULTI_DIR = os.path.join(BASE_PATH, "multivariate")

In [2]:
def get_dataset_names(path: str) -> List[str]:
    """
    Scans the specified directory and returns a list of dataset names.
    Ignores hidden files and directories.
    """
    return [f.name for f in os.scandir(path) if f.is_dir() and not f.name.startswith('.')]


def load_dataset_shapes(dataset_name: str, base_dir: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Loads strictly the data arrays from pickle files to avoid loading heavy models or explainers.
    """
    ds_path = os.path.join(base_dir, dataset_name)

    with open(os.path.join(ds_path, 'trainX.pickle'), 'rb') as f:
        trainX = pickle.load(f)
    with open(os.path.join(ds_path, 'trainy.pickle'), 'rb') as f:
        trainy = pickle.load(f)
    with open(os.path.join(ds_path, 'testX.pickle'), 'rb') as f:
        testX = pickle.load(f)
    with open(os.path.join(ds_path, 'testy.pickle'), 'rb') as f:
        testy = pickle.load(f)

    return trainX, trainy, testX, testy

In [3]:
def extract_descriptive_stats(
    trainX: np.ndarray,
    trainy: np.ndarray,
    testX: np.ndarray,
    dataset_name: str,
    ds_type: str
) -> Dict[str, Any]:
    """
    Extracts structural and descriptive statistics from standard time series arrays.
    """
    train_size, timesteps, channels = trainX.shape
    test_size = testX.shape[0]
    num_classes = trainy.shape[1]

    nan_train = np.isnan(trainX).sum()
    nan_test = np.isnan(testX).sum()

    return {
        "Dataset": dataset_name,
        "Type": ds_type,
        "Train_Size": train_size,
        "Test_Size": test_size,
        "Total_Size": train_size + test_size,
        "Timesteps": timesteps,
        "Channels": channels,
        "Classes": num_classes,
        "NaNs_Train": nan_train,
        "NaNs_Test": nan_test
    }

In [4]:
def generate_statistics_report(uni_dir: str, multi_dir: str) -> pd.DataFrame:
    """
    Orchestrates the extraction process across univariate and multivariate datasets.
    Returns a compiled pandas DataFrame with all statistics.
    """
    stats = []

    for ds in get_dataset_names(uni_dir):
        trainX, trainy, testX, _ = load_dataset_shapes(ds, uni_dir)
        ds_stats = extract_descriptive_stats(trainX, trainy, testX, ds, "Univariate")
        stats.append(ds_stats)

    for ds in get_dataset_names(multi_dir):
        trainX, trainy, testX, _ = load_dataset_shapes(ds, multi_dir)
        ds_stats = extract_descriptive_stats(trainX, trainy, testX, ds, "Multivariate")
        stats.append(ds_stats)

    return pd.DataFrame(stats)

In [13]:
# Execute extraction pipeline
df_stats = generate_statistics_report(UNI_DIR, MULTI_DIR)

In [14]:
df_stats

,Dataset,Type,Train_Size,Test_Size,Total_Size,Timesteps,Channels,Classes,NaNs_Train,NaNs_Test
0,DodgerLoopGame,Univariate,118,40,158,288,1,2,0,0
1,ProximalPhalanxTW,Univariate,453,152,605,80,1,6,0,0
2,DodgerLoopDay,Univariate,118,40,158,288,1,7,0,0
3,ECGFiveDays,Univariate,663,221,884,136,1,2,0,0
4,UMD,Univariate,135,45,180,150,1,3,0,0
...,...,...,...,...,...,...,...,...,...,...
99,Cricket,Multivariate,135,45,180,1197,6,12,0,0
100,LSST,Multivariate,3693,1232,4925,36,6,14,0,0
101,UWaveGestureLibrary,Multivariate,330,110,440,315,3,8,0,0
102,RacketSports,Multivariate,227,76,303,30,6,4,0,0


In [15]:
# Sort for better readability in the paper
df_stats.sort_values(by=["Type", "Total_Size"], ascending=[False, False], inplace=True)
df_stats = df_stats[df_stats["Dataset"] != "ChlorineConcentration"]

# Save to required formats
csv_path = os.path.join(BASE_PATH, "datasets_descriptive_statistics.csv")
latex_path = os.path.join(BASE_PATH, "datasets_descriptive_statistics.tex")

df_stats.to_csv(csv_path, index=False)
df_stats.to_latex(latex_path, index=False)

# Display the dataframe in Jupyter
df_stats

,Dataset,Type,Train_Size,Test_Size,Total_Size,Timesteps,Channels,Classes,NaNs_Train,NaNs_Test
37,Crop,Univariate,18000,6000,24000,46,1,24,0,0
24,ElectricDevices,Univariate,12477,4160,16637,96,1,7,0,0
53,Wafer,Univariate,5373,1791,7164,152,1,2,0,0
7,TwoPatterns,Univariate,3750,1250,5000,128,1,4,0,0
27,ECG5000,Univariate,3750,1250,5000,140,1,5,0,0
...,...,...,...,...,...,...,...,...,...,...
95,Epilepsy,Multivariate,206,69,275,206,3,4,0,0
87,HandMovementDirection,Multivariate,175,59,234,400,10,4,0,0
99,Cricket,Multivariate,135,45,180,1197,6,12,0,0
86,BasicMotions,Multivariate,60,20,80,100,6,4,0,0


In [16]:
def compute_benchmark_summary(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Computes summary statistics (min, median, max) for dataset characteristics
    to be used inline within the manuscript text.
    """
    summary = {
        "Total_Datasets": len(df),
        "Size_Min": df["Total_Size"].min(),
        "Size_Median": df["Total_Size"].median(),
        "Size_Max": df["Total_Size"].max(),
        "Length_Min": df["Timesteps"].min(),
        "Length_Median": df["Timesteps"].median(),
        "Length_Max": df["Timesteps"].max(),
        "Classes_Min": df["Classes"].min(),
        "Classes_Median": df["Classes"].median(),
        "Classes_Max": df["Classes"].max(),
        "Multi_Channels_Max": df[df["Type"] == "Multivariate"]["Channels"].max()
    }

    # Generate a draft sentence for the paper
    draft_text = (
        f"To provide a broad evaluation, the selected {summary['Total_Datasets']} datasets vary significantly in scale and complexity: "
        f"total instance counts range from {summary['Size_Min']} to {summary['Size_Max']} (median: {int(summary['Size_Median'])}), "
        f"time-series lengths span from {summary['Length_Min']} to {summary['Length_Max']} time steps (median: {int(summary['Length_Median'])}), "
        f"and the number of target classes varies between {summary['Classes_Min']} and {summary['Classes_Max']} (median: {int(summary['Classes_Median'])}). "
        f"While univariate datasets contain a single channel, the multivariate tasks include up to {summary['Multi_Channels_Max']} dimensions."
    )

    print("--- DRAFT FOR THE MANUSCRIPT ---")
    print(draft_text)

    return summary

In [17]:
df_stats.info()

<class 'pandas.core.frame.DataFrame'>
Index: 103 entries, 37 to 97
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Dataset     103 non-null    object
 1   Type        103 non-null    object
 2   Train_Size  103 non-null    int64 
 3   Test_Size   103 non-null    int64 
 4   Total_Size  103 non-null    int64 
 5   Timesteps   103 non-null    int64 
 6   Channels    103 non-null    int64 
 7   Classes     103 non-null    int64 
 8   NaNs_Train  103 non-null    int64 
 9   NaNs_Test   103 non-null    int64 
dtypes: int64(8), object(2)
memory usage: 12.9+ KB


In [18]:
df_stats

,Dataset,Type,Train_Size,Test_Size,Total_Size,Timesteps,Channels,Classes,NaNs_Train,NaNs_Test
37,Crop,Univariate,18000,6000,24000,46,1,24,0,0
24,ElectricDevices,Univariate,12477,4160,16637,96,1,7,0,0
53,Wafer,Univariate,5373,1791,7164,152,1,2,0,0
7,TwoPatterns,Univariate,3750,1250,5000,128,1,4,0,0
27,ECG5000,Univariate,3750,1250,5000,140,1,5,0,0
...,...,...,...,...,...,...,...,...,...,...
95,Epilepsy,Multivariate,206,69,275,206,3,4,0,0
87,HandMovementDirection,Multivariate,175,59,234,400,10,4,0,0
99,Cricket,Multivariate,135,45,180,1197,6,12,0,0
86,BasicMotions,Multivariate,60,20,80,100,6,4,0,0


In [19]:
stats_summary = compute_benchmark_summary(df_stats)

--- DRAFT FOR THE MANUSCRIPT ---
To provide a broad evaluation, the selected 103 datasets vary significantly in scale and complexity: total instance counts range from 30 to 24000 (median: 553), time-series lengths span from 8 to 1751 time steps (median: 235), and the number of target classes varies between 2 and 60 (median: 3). While univariate datasets contain a single channel, the multivariate tasks include up to 144 dimensions.
